
# 🧭 Week 3 — Pipelines, Modularity & Debugging with DSPY

<a href="https://colab.research.google.com/github/tulane-intro-ai-engineering/main/blob/main/lectures/week3_pipelines_and_dspy.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/>
</a>

*How AI engineers move from single prompts to reliable, modular LLM systems.*

---

## 🎯 Learning Objectives

By the end of this week, you should be able to:
1. Explain what a pipeline is and why modularity matters for reliability.  
2. Distinguish between monolithic and multi-step prompt designs.  
3. Configure and run a basic LLM in `dspy` using the OpenAI API.  
4. Build, inspect, and debug simple pipelines step-by-step.  
5. Use `dspy.inspect()` and logging to understand what’s actually happening inside an LLM pipeline.


# 📅 Day 1 — From Prompts to Pipelines


## Section 1 — Lab L2 Recap: What Changed When Temperature Increased?

**Guiding Question:**  
> What did you notice when temperature increased? Was the model more creative, or less reliable?

Discuss:
- Higher temperature → more diverse completions.
- Lower temperature → more deterministic and stable outputs.



## Section 2 — Motivating Example: Why Modularity?

**Question:**  
> Why can’t we just write one giant prompt and call it a day?

We'll explore examples of LLM success and failure:
- **Success:** multi-step reasoning task (summarize → critique → rewrite).
- **Failure:** same task with one unstructured prompt.

**Reflection:**  
> What might have gone wrong?  
> How could you design the system to isolate each step?



## Section 3 — Concept: What Is a Pipeline?

A **pipeline** is a sequence of processing steps that move data forward.  
Each stage transforms or filters information before handing it off.

**Mathematical intuition:**

$$
P(Y|X) = \prod_{i=1}^{n} P(Y_i | Y_{<i}, X)
$$



In [ ]:
# @title Visualization: Conceptual Flow of an LLM Pipeline
import matplotlib.pyplot as plt

stages = ["Input", "Extract", "Simplify", "Summarize", "Output"]
plt.figure(figsize=(8,1))
for i,s in enumerate(stages):
    plt.text(i, 0.5, s, ha='center', va='center',
             bbox=dict(facecolor='lightblue', edgecolor='black'))
plt.axis('off')
plt.title("Conceptual Flow of an LLM Pipeline")
plt.show()



## Section 4 — Mini DSPY Preview: The Simplest Predictor

**Guiding Question:**  
> What does it mean to describe a prompt as a *function*?


In [ ]:
# collapsed
import dspy

predict = dspy.Predict("question -> answer")
result = predict(question="What is the capital of France?")
print(result.answer)


In [ ]:
# collapsed
dspy.inspect(predict(question="What is the capital of France?"))



## Section 5 — Debugging by Decomposition

**Guiding Question:**  
> What happens when a single step in a multi-step system fails?


In [ ]:
# collapsed
import random

steps = ["extract", "simplify", "summarize"]
for s in steps:
    success = random.random() > 0.2
    print(f"{s}: {'✅ success' if success else '❌ failure'}")



## Section 6 — Activity: Sketch a Real AI Pipeline

In pairs, pick an AI system (e.g., ChatGPT, Grammarly, Copilot).  
Sketch its internal pipeline (3–4 boxes max):  
*Input Handling → Intent Detection → LLM → Postprocessing*



## Section 7 — 5-Minute Concept Quiz

1. Why is modularity useful in AI pipelines?  
2. What is “failure localization”?  
3. Why might debugging be harder in a single giant prompt?  
4. How does `dspy` help with modularity?  
5. What does `dspy.inspect()` show?



## Section 8 — Unifying Diagram v2

```mermaid
graph TD
  U["User Input"] --> IH["Input Handling"]
  IH --> S1["Step 1: Extract Info"]
  S1 --> S2["Step 2: Simplify"]
  S2 --> LLM["Core LLM"]
  LLM --> OP["Output Processing"]
  OP --> M["Monitoring"]
```



<details>
<summary>🧑‍🏫 <b>Instructor Notes (Day 1)</b></summary>

- Timing: 10 + 10 + 15 + 15 + 15 + 5 + 5  
- Use `dspy.inspect()` live to reveal prompts.  
- Encourage students to draw diagrams and share debugging analogies.  
- Wrap with a quiz and transition: “Next class, we’ll *build* pipelines in DSPY.”

</details>


# 📅 Day 2 — Building and Debugging Pipelines with DSPY

In [ ]:
# @title 🔧 Setup DSPY with OpenAI
import dspy, os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("🔑 Enter your OpenAI key: ")
dspy.configure(model="gpt-4o-mini", api_key=os.environ["OPENAI_API_KEY"])
print("✅ DSPY configured with OpenAI")



## Section 2 — Typed Predictors


In [ ]:
# collapsed
sentiment = dspy.Predict("sentence -> sentiment: bool")
result = sentiment(sentence="I love debugging pipelines!")
print("Sentiment:", result.sentiment)


In [ ]:
# collapsed
qa = dspy.Predict("question -> reasoning, answer")
resp = qa(question="Why do leaves change color in autumn?")
print("Reasoning:", resp.reasoning)
print("Answer:", resp.answer)


In [ ]:
# collapsed
extract = dspy.Predict("article -> key_points")
simplify = dspy.Predict("key_points -> summary")
tag = dspy.Predict("summary -> tags: list[str]")

def summarize_pipeline(article):
    key_points = extract(article=article).key_points
    summary = simplify(key_points=key_points).summary
    tags = tag(summary=summary).tags
    return summary, tags

article = "Photosynthesis converts light energy into chemical energy..."
print(summarize_pipeline(article))


In [ ]:
# collapsed
def safe_run(predictor, **kwargs):
    try:
        result = predictor(**kwargs)
        if not result:
            raise ValueError("Empty output")
        return result
    except Exception as e:
        print(f"⚠️ Error in {predictor}: {e}")
        return None

result = safe_run(extract, article="Invalid input")


In [ ]:
# collapsed
import pandas as pd

trace = [
    {"step": "extract", "status": "✅", "details": "3 key points"},
    {"step": "simplify", "status": "✅", "details": "Readable summary"},
    {"step": "tag", "status": "⚠️", "details": "Low confidence tags"}
]

pd.DataFrame(trace)


In [ ]:
# collapsed
class Summarizer(dspy.Module):
    def __init__(self):
        self.extract = dspy.Predict("article -> key_points")
        self.simplify = dspy.Predict("key_points -> summary")
    def forward(self, article):
        return self.simplify(key_points=self.extract(article=article).key_points).summary

module = Summarizer()
print(module("The sun provides energy for plants..."))


In [ ]:
# collapsed
logs = []
article = "AI systems are more interpretable when modular."
summary = module(article)
logs.append({"input": article, "output": summary})

import pandas as pd
pd.DataFrame(logs)



## Section 8 — Wrap-Up Discussion

- Compare: Monolithic vs modular prompts.  
- Review: `dspy.inspect()`, error handling, logging, and modularization.  
- Preview: Next week—**Embeddings** and **semantic retrieval**.



<details>
<summary>🧑‍🏫 <b>Instructor Notes (Day 2)</b></summary>

- Timing: 10 + 10 + 15 + 20 + 10 + 5 + 5  
- Have students run `dspy.inspect()` and `safe_run()` interactively.  
- Encourage experimentation: intentionally break steps and observe errors.  
- Close by linking debugging practices to system reliability.

</details>
